# Multi-Agent Podcast Research Pipeline


### Imports

In [ ]:
%pip install -q "crewai[tools]>=1.6.1,<2" python-dotenv exa-py

import os
from datetime import datetime

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai_tools import EXASearchTool
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()
os.makedirs("task_outputs", exist_ok=True)

### Step 1: Podcast Researcher Agent + Task + Tool


In [ ]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about the given topic",
    # Prompt Engineering: Role/Persona Priming
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=False,
)

# Prompt Engineering: Specificity / Constraint Engineering
research_task = Task(
    description=(
        "Create a research brief for a podcast episode about AI Agents taking over job roles in 2026"
        "Search for 5 recent and important developments. "
        "For each, include: title, short summary, and why it matters for listeners."
    ),
    expected_output="A podcast research brief with 5 findings, each with title, summary, and listener angle.",
    agent=researcher,
    output_file="task_outputs/step_1/step1_research.md",
)

single_agent_crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)

_ = single_agent_crew.kickoff()
print("Research saved to task_outputs/step_1/step1_research.md")

### Step 2: Production Parameters


In [ ]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=False,
    max_iter=5,
    max_rpm=12,
    max_execution_time=300,
    respect_context_window=True,
)

print(f"Researcher ready: max_iter={researcher.max_iter}, max_rpm={researcher.max_rpm}")

### Step 3: Reporting Analyst Agent + Context Passing — The Multi-Agent Pattern


In [ ]:
reporting_analyst = Agent(
    role="AI Podcast Reporting Analyst",
    goal="Turn raw research findings about {topic} into a structured podcast briefing report",
    backstory=(
        "You are a meticulous analyst who turns messy research into structured reports. "
        "You organize findings into sections that are easy for podcast hosts to understand and riff on."
    ),
    llm=llm,
    verbose=False,
    max_iter=4,
    max_rpm=10,
    max_execution_time=240,
    respect_context_window=True,
)

In [ ]:
research_task = Task(
    description=(
        "Conduct thorough podcast research about {topic}. "
        "Find recent releases, articles, products, debates, and progress. "
        "Use the current date context: {current_month} {current_year}."
    ),
    expected_output=(
        "A list of 10 relevant findings about {topic}. For each finding include: "
        "title, short summary, why it matters for listeners, and source if available."
    ),
    agent=researcher,
    output_file="task_outputs/step_3/step3_research.md",
)

reporting_task = Task(
    description=(
        "Review the research findings and organize them into a complete podcast briefing report. "
        "Group related findings into sections and explain the narrative arc for the episode."
    ),
    expected_output=(
        "A markdown report with: episode angle, key sections with findings grouped thematically, "
        "important facts, listener takeaways, and suggested host questions."
    ),
    agent=reporting_analyst,
    context=[
        research_task
    ],  # Prompt Chaining: research_task output feeds directly into this task
    output_file="task_outputs/step_3/step3_briefing.md",
)

## Step 4: The Crew — Orchestrate the Pipeline


In [ ]:
podcast_crew = Crew(
    agents=[researcher, reporting_analyst],
    tasks=[research_task, reporting_task],
    process=Process.sequential,
    verbose=True,
)

inputs = {
    "topic": "AI Agents taking over job roles",
    "current_month": datetime.now().strftime("%B"),
    "current_year": str(datetime.now().year),
}

_ = podcast_crew.kickoff(inputs=inputs)
print("Pipeline complete — outputs saved to task_outputs/")

## Step 5: Swap the Topic


In [ ]:
# Uncomment to run a second episode on a different topic.
# podcast_crew.kickoff(inputs={
#     "topic": "AI agents for software engineering",
#     "current_month": datetime.now().strftime("%B"),
#     "current_year": str(datetime.now().year),
# })